# SOPR Capitulation + Distance Filter - Final Strategy

**The Discovery:** Excellent entries happen when price is within 30% of 52-week high (corrections in uptrends). Terrible entries happen when price is >50% below high (bear market capitulation).

**Strategy:**
- Entry: SOPR < 1 AND STH SOPR < 1 AND price within 30% of 52-week high
- Exit: Trailing stop (8% stop loss, 12% trail, activates at 5% profit)

**Goal:** Beat the baseline 54% walk-forward beat rate

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Let's go! 🚀")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")

df = sopr.join(sopr_sth, how='inner').join(price, how='inner').sort_index()
df = df[df.index >= '2018-12-15']

close = df['price']
print(f"Data: {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")

In [ ]:
# Calculate distance from 52-week high
df['high_52w'] = df['price'].rolling(365, min_periods=1).max()
df['dist_from_high'] = (df['price'] / df['high_52w'] - 1) * 100  # Negative = below high

print(f"Distance from 52w high stats:")
print(f"  Min: {df['dist_from_high'].min():.1f}%")
print(f"  Max: {df['dist_from_high'].max():.1f}%")
print(f"  Median: {df['dist_from_high'].median():.1f}%")

In [ ]:
# Base entry signal (SOPR double capitulation)
both_below_1 = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
base_entries = both_below_1 & ~both_below_1.shift(1).fillna(False)

print(f"Base signals (no filter): {base_entries.sum()}")

# Filtered entry signal (within 30% of high)
distance_filter = df['dist_from_high'] > -30  # Within 30% of high
filtered_entries = base_entries & distance_filter

print(f"Filtered signals (< 30% from high): {filtered_entries.sum()}")
print(f"Removed: {base_entries.sum() - filtered_entries.sum()} signals")

In [ ]:
# Visualize filter
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3],
                    subplot_titles=['Price with Entry Signals', 'Distance from 52w High'])

# Price
fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price',
                         line=dict(color='blue', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['high_52w'], name='52w High',
                         line=dict(color='gray', width=1, dash='dot')), row=1, col=1)

# Base entries (removed)
removed = base_entries & ~filtered_entries
removed_dates = removed[removed].index
fig.add_trace(go.Scatter(
    x=removed_dates, y=df.loc[removed_dates, 'price'],
    mode='markers', marker=dict(symbol='x', size=10, color='red'),
    name=f'Removed ({len(removed_dates)})'
), row=1, col=1)

# Filtered entries (kept)
kept_dates = filtered_entries[filtered_entries].index
fig.add_trace(go.Scatter(
    x=kept_dates, y=df.loc[kept_dates, 'price'],
    mode='markers', marker=dict(symbol='triangle-up', size=10, color='green'),
    name=f'Kept ({len(kept_dates)})'
), row=1, col=1)

# Distance from high
fig.add_trace(go.Scatter(x=df.index, y=df['dist_from_high'], name='Dist from High',
                         line=dict(color='purple', width=1)), row=2, col=1)
fig.add_hline(y=-30, line_dash='dash', line_color='red', row=2, col=1,
              annotation_text='-30% threshold')

fig.update_layout(height=700, title_text='Entry Filter: Keep signals within 30% of 52w High')
fig.update_yaxes(type='log', row=1, col=1)
fig.show()

---
## Trailing Stop Backtester

In [ ]:
def backtest_trailing_stop(
    close: pd.Series,
    entries: pd.Series,
    stop_loss: float = 0.08,
    trailing_stop: float = 0.12,
    min_profit_to_trail: float = 0.05,
    max_hold_days: int = 180,
):
    """Backtest with trailing stop."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = close.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        initial_stop = entry_price * (1 - stop_loss)
        current_stop = initial_stop
        is_trailing = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(close)):
            current_date = close.index[j]
            current_price = close.iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            current_pnl = (current_price - entry_price) / entry_price
            
            if not is_trailing and current_pnl >= min_profit_to_trail:
                is_trailing = True
            
            if is_trailing:
                trailing_stop_level = peak_price * (1 - trailing_stop)
                if trailing_stop_level > current_stop:
                    current_stop = trailing_stop_level
            
            if current_price <= current_stop:
                exit_date = current_date
                exit_price = current_stop
                exit_reason = 'trailing_stop' if is_trailing else 'stop_loss'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = close.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'entry_price': entry_price,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

---
## Compare: With Filter vs Without Filter

In [ ]:
# Strategy parameters
STOP_LOSS = 0.08
TRAILING_STOP = 0.12
MIN_PROFIT = 0.05

# Backtest both
trades_no_filter = backtest_trailing_stop(close, base_entries, STOP_LOSS, TRAILING_STOP, MIN_PROFIT)
trades_with_filter = backtest_trailing_stop(close, filtered_entries, STOP_LOSS, TRAILING_STOP, MIN_PROFIT)

print(f"Without filter: {len(trades_no_filter)} trades")
print(f"With filter: {len(trades_with_filter)} trades")

In [ ]:
def calc_stats(trades):
    """Calculate strategy stats."""
    if len(trades) == 0:
        return {}
    
    total_return = (1 + trades['pnl_pct']).prod() - 1
    win_rate = (trades['pnl_pct'] > 0).mean()
    avg_win = trades[trades['pnl_pct'] > 0]['pnl_pct'].mean() if (trades['pnl_pct'] > 0).any() else 0
    avg_loss = trades[trades['pnl_pct'] <= 0]['pnl_pct'].mean() if (trades['pnl_pct'] <= 0).any() else 0
    
    gross_win = trades[trades['pnl_pct'] > 0]['pnl_pct'].sum()
    gross_loss = abs(trades[trades['pnl_pct'] <= 0]['pnl_pct'].sum())
    profit_factor = gross_win / gross_loss if gross_loss > 0 else np.inf
    
    return {
        'n_trades': len(trades),
        'total_return': total_return,
        'win_rate': win_rate,
        'avg_win': avg_win,
        'avg_loss': avg_loss,
        'profit_factor': profit_factor,
        'avg_days': trades['days_held'].mean()
    }

stats_no_filter = calc_stats(trades_no_filter)
stats_with_filter = calc_stats(trades_with_filter)

print("\nIN-SAMPLE COMPARISON")
print("="*70)
print(f"{'Metric':<20} {'No Filter':>20} {'With Filter':>20}")
print("-"*70)
print(f"{'Trades':<20} {stats_no_filter['n_trades']:>20} {stats_with_filter['n_trades']:>20}")
print(f"{'Total Return':<20} {stats_no_filter['total_return']*100:>19.0f}% {stats_with_filter['total_return']*100:>19.0f}%")
print(f"{'Win Rate':<20} {stats_no_filter['win_rate']*100:>19.0f}% {stats_with_filter['win_rate']*100:>19.0f}%")
print(f"{'Avg Win':<20} {stats_no_filter['avg_win']*100:>19.1f}% {stats_with_filter['avg_win']*100:>19.1f}%")
print(f"{'Avg Loss':<20} {stats_no_filter['avg_loss']*100:>19.1f}% {stats_with_filter['avg_loss']*100:>19.1f}%")
print(f"{'Profit Factor':<20} {stats_no_filter['profit_factor']:>20.2f} {stats_with_filter['profit_factor']:>20.2f}")

In [ ]:
# Trade details for filtered strategy
print("\nFILTERED STRATEGY TRADES")
print("="*100)

display_trades = trades_with_filter.copy()
display_trades['entry_date'] = pd.to_datetime(display_trades['entry_date']).dt.strftime('%Y-%m-%d')
display_trades['exit_date'] = pd.to_datetime(display_trades['exit_date']).dt.strftime('%Y-%m-%d')
display_trades['entry_price'] = display_trades['entry_price'].round(0).astype(int)
display_trades['exit_price'] = display_trades['exit_price'].round(0).astype(int)
display_trades['pnl_pct'] = (display_trades['pnl_pct'] * 100).round(1)

print(display_trades.to_string(index=False))

---
## Walk-Forward Validation (The Real Test)

In [ ]:
def walk_forward_test(entries, close, train_days=365, test_days=90, step_days=90):
    """Walk-forward validation."""
    wf_results = []
    
    total_days = len(close)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        if test_end <= test_start:
            break
        
        test_close = close.iloc[test_start:test_end]
        test_entries = entries.iloc[test_start:test_end]
        
        trades = backtest_trailing_stop(
            close=test_close,
            entries=test_entries,
            stop_loss=STOP_LOSS,
            trailing_stop=TRAILING_STOP,
            min_profit_to_trail=MIN_PROFIT,
            max_hold_days=180
        )
        
        if len(trades) > 0:
            strat_return = (1 + trades['pnl_pct']).prod() - 1
            n_trades = len(trades)
        else:
            strat_return = 0
            n_trades = 0
        
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        wf_results.append({
            'fold': fold,
            'period': close.index[test_start].strftime('%Y-%m'),
            'n_trades': n_trades,
            'strat_return': strat_return,
            'hold_return': hold_return,
            'excess': strat_return - hold_return,
            'beat_hold': strat_return > hold_return
        })
    
    return pd.DataFrame(wf_results)

In [ ]:
# Walk-forward both strategies
wf_no_filter = walk_forward_test(base_entries, close)
wf_with_filter = walk_forward_test(filtered_entries, close)

print("WALK-FORWARD RESULTS - NO FILTER")
print("="*80)
for _, row in wf_no_filter.iterrows():
    status = '✓' if row['beat_hold'] else '✗'
    print(f"Fold {row['fold']:2d}: {row['period']} | {row['n_trades']:2d} trades | "
          f"Strat: {row['strat_return']*100:+6.1f}% | B&H: {row['hold_return']*100:+6.1f}% | {status}")

print(f"\nBeat Rate: {wf_no_filter['beat_hold'].mean()*100:.0f}%")

In [ ]:
print("\nWALK-FORWARD RESULTS - WITH FILTER")
print("="*80)
for _, row in wf_with_filter.iterrows():
    status = '✓' if row['beat_hold'] else '✗'
    print(f"Fold {row['fold']:2d}: {row['period']} | {row['n_trades']:2d} trades | "
          f"Strat: {row['strat_return']*100:+6.1f}% | B&H: {row['hold_return']*100:+6.1f}% | {status}")

print(f"\nBeat Rate: {wf_with_filter['beat_hold'].mean()*100:.0f}%")

In [ ]:
# Side by side comparison
print("\n" + "="*70)
print("WALK-FORWARD COMPARISON")
print("="*70)
print(f"{'Metric':<30} {'No Filter':>15} {'With Filter':>15}")
print("-"*70)
print(f"{'Total Folds':<30} {len(wf_no_filter):>15} {len(wf_with_filter):>15}")
print(f"{'Folds with Trades':<30} {(wf_no_filter['n_trades'] > 0).sum():>15} {(wf_with_filter['n_trades'] > 0).sum():>15}")
print(f"{'Beat Buy & Hold Rate':<30} {wf_no_filter['beat_hold'].mean()*100:>14.0f}% {wf_with_filter['beat_hold'].mean()*100:>14.0f}%")
print(f"{'Avg Excess Return':<30} {wf_no_filter['excess'].mean()*100:>+14.1f}% {wf_with_filter['excess'].mean()*100:>+14.1f}%")
print(f"{'Avg Strategy Return':<30} {wf_no_filter['strat_return'].mean()*100:>14.1f}% {wf_with_filter['strat_return'].mean()*100:>14.1f}%")

In [ ]:
# Visualize walk-forward comparison
fig = make_subplots(rows=1, cols=2, subplot_titles=['Beat Rate', 'Avg Excess Return'])

# Beat rate
fig.add_trace(go.Bar(
    x=['No Filter', 'With Filter'],
    y=[wf_no_filter['beat_hold'].mean()*100, wf_with_filter['beat_hold'].mean()*100],
    marker_color=['steelblue', 'green'],
    text=[f"{wf_no_filter['beat_hold'].mean()*100:.0f}%", f"{wf_with_filter['beat_hold'].mean()*100:.0f}%"],
    textposition='outside'
), row=1, col=1)

fig.add_hline(y=50, line_dash='dash', line_color='red', row=1, col=1)

# Avg excess
fig.add_trace(go.Bar(
    x=['No Filter', 'With Filter'],
    y=[wf_no_filter['excess'].mean()*100, wf_with_filter['excess'].mean()*100],
    marker_color=['steelblue', 'green'],
    text=[f"{wf_no_filter['excess'].mean()*100:+.1f}%", f"{wf_with_filter['excess'].mean()*100:+.1f}%"],
    textposition='outside'
), row=1, col=2)

fig.add_hline(y=0, line_dash='dash', line_color='red', row=1, col=2)

fig.update_layout(height=400, showlegend=False, title_text='Walk-Forward: Filter vs No Filter')
fig.show()

---
## Equity Curves

In [ ]:
# Build equity curves
initial_capital = 100000

# No filter equity
equity_no_filter = initial_capital * (1 + trades_no_filter['pnl_pct']).cumprod()

# With filter equity
equity_with_filter = initial_capital * (1 + trades_with_filter['pnl_pct']).cumprod()

# Buy and hold
bh_final = initial_capital * (close.iloc[-1] / close.iloc[0])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=trades_no_filter['exit_date'], y=equity_no_filter.values,
    mode='lines+markers', name='No Filter',
    line=dict(color='steelblue', width=2)
))

fig.add_trace(go.Scatter(
    x=trades_with_filter['exit_date'], y=equity_with_filter.values,
    mode='lines+markers', name='With Filter',
    line=dict(color='green', width=2)
))

fig.add_hline(y=bh_final, line_dash='dash', line_color='gray',
              annotation_text=f'Buy & Hold: ${bh_final:,.0f}')
fig.add_hline(y=initial_capital, line_dash='dot', line_color='black')

fig.update_layout(
    title='Equity Curves: Filter vs No Filter',
    yaxis_title='Portfolio Value ($)',
    height=500
)
fig.show()

print(f"\nFinal Values:")
print(f"  No Filter: ${equity_no_filter.iloc[-1]:,.0f}")
print(f"  With Filter: ${equity_with_filter.iloc[-1]:,.0f}")
print(f"  Buy & Hold: ${bh_final:,.0f}")

---
## Test Different Distance Thresholds

In [ ]:
# Test different distance thresholds
thresholds = [20, 25, 30, 35, 40, 50]
threshold_results = []

for thresh in thresholds:
    filter_mask = df['dist_from_high'] > -thresh
    test_entries = base_entries & filter_mask
    
    # In-sample
    trades = backtest_trailing_stop(close, test_entries, STOP_LOSS, TRAILING_STOP, MIN_PROFIT)
    stats = calc_stats(trades) if len(trades) > 0 else {'n_trades': 0, 'total_return': 0, 'win_rate': 0, 'profit_factor': 0}
    
    # Walk-forward
    wf = walk_forward_test(test_entries, close)
    
    threshold_results.append({
        'threshold': f'{thresh}%',
        'n_trades': stats['n_trades'],
        'total_return': stats['total_return'],
        'win_rate': stats.get('win_rate', 0),
        'profit_factor': stats.get('profit_factor', 0),
        'wf_beat_rate': wf['beat_hold'].mean(),
        'wf_avg_excess': wf['excess'].mean()
    })

thresh_df = pd.DataFrame(threshold_results)

print("\nDISTANCE THRESHOLD COMPARISON")
print("="*100)
print(f"{'Threshold':<12} {'Trades':>8} {'Return':>12} {'Win Rate':>10} {'PF':>8} {'WF Beat':>10} {'WF Excess':>12}")
print("-"*100)
for _, row in thresh_df.iterrows():
    print(f"{row['threshold']:<12} {row['n_trades']:>8} {row['total_return']*100:>11.0f}% "
          f"{row['win_rate']*100:>9.0f}% {row['profit_factor']:>8.2f} "
          f"{row['wf_beat_rate']*100:>9.0f}% {row['wf_avg_excess']*100:>+11.1f}%")

In [ ]:
# Visualize threshold comparison
fig = make_subplots(rows=1, cols=2, subplot_titles=['Walk-Forward Beat Rate', 'Walk-Forward Avg Excess'])

fig.add_trace(go.Bar(
    x=thresh_df['threshold'],
    y=thresh_df['wf_beat_rate']*100,
    marker_color='steelblue',
    text=[f"{x:.0f}%" for x in thresh_df['wf_beat_rate']*100],
    textposition='outside'
), row=1, col=1)

fig.add_hline(y=50, line_dash='dash', line_color='red', row=1, col=1)

colors = ['green' if x > 0 else 'red' for x in thresh_df['wf_avg_excess']]
fig.add_trace(go.Bar(
    x=thresh_df['threshold'],
    y=thresh_df['wf_avg_excess']*100,
    marker_color=colors,
    text=[f"{x*100:+.1f}%" for x in thresh_df['wf_avg_excess']],
    textposition='outside'
), row=1, col=2)

fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=2)

fig.update_layout(height=400, showlegend=False, title_text='Distance Threshold Impact on Walk-Forward')
fig.show()

---
## Final Summary

In [ ]:
# Find best threshold
best_idx = thresh_df['wf_beat_rate'].idxmax()
best_thresh = thresh_df.loc[best_idx]

print("\n" + "="*70)
print("FINAL STRATEGY SUMMARY")
print("="*70)

print(f"\n📊 STRATEGY")
print(f"   Entry: SOPR < 1 AND STH SOPR < 1")
print(f"   Filter: Price within {best_thresh['threshold']} of 52-week high")
print(f"   Exit: Trailing Stop (SL {STOP_LOSS*100:.0f}%, Trail {TRAILING_STOP*100:.0f}%, activate at {MIN_PROFIT*100:.0f}%)")

print(f"\n📈 IN-SAMPLE RESULTS")
print(f"   Trades: {best_thresh['n_trades']:.0f}")
print(f"   Total Return: {best_thresh['total_return']*100:.0f}%")
print(f"   Win Rate: {best_thresh['win_rate']*100:.0f}%")
print(f"   Profit Factor: {best_thresh['profit_factor']:.2f}")

print(f"\n🔍 WALK-FORWARD RESULTS")
print(f"   Beat Buy & Hold: {best_thresh['wf_beat_rate']*100:.0f}%")
print(f"   Avg Excess Return: {best_thresh['wf_avg_excess']*100:+.1f}%")

baseline_beat = wf_no_filter['beat_hold'].mean()
improvement = best_thresh['wf_beat_rate'] - baseline_beat

print(f"\n🎯 IMPROVEMENT OVER BASELINE")
print(f"   Baseline (no filter): {baseline_beat*100:.0f}%")
print(f"   With filter: {best_thresh['wf_beat_rate']*100:.0f}%")
print(f"   Improvement: {improvement*100:+.0f}%")

if best_thresh['wf_beat_rate'] > 0.55:
    verdict = "✅ STRATEGY HAS EDGE"
elif best_thresh['wf_beat_rate'] > 0.50:
    verdict = "⚠️ MARGINAL EDGE"
else:
    verdict = "❌ NO EDGE"

print(f"\n🏆 VERDICT: {verdict}")
print("\n" + "="*70)

In [ ]:
# Save results
import json

final_results = {
    'signal': 'sopr_double_capitulation',
    'entry': 'SOPR < 1 AND STH_SOPR < 1',
    'filter': f'price within {best_thresh["threshold"]} of 52w high',
    'exit_strategy': 'trailing_stop',
    'exit_params': {
        'stop_loss': STOP_LOSS,
        'trailing_stop': TRAILING_STOP,
        'min_profit_to_trail': MIN_PROFIT
    },
    'in_sample': {
        'n_trades': int(best_thresh['n_trades']),
        'total_return': float(best_thresh['total_return']),
        'win_rate': float(best_thresh['win_rate']),
        'profit_factor': float(best_thresh['profit_factor'])
    },
    'walk_forward': {
        'beat_hold_pct': float(best_thresh['wf_beat_rate']),
        'avg_excess': float(best_thresh['wf_avg_excess'])
    },
    'baseline_comparison': {
        'baseline_beat_rate': float(baseline_beat),
        'improvement': float(improvement)
    },
    'all_thresholds_tested': thresh_df.to_dict('records')
}

with open('../data/sopr_final_strategy_results.json', 'w') as f:
    json.dump(final_results, f, indent=2, default=str)

print("Saved to ../data/sopr_final_strategy_results.json")